# E-GACT v4 — AIM Final Reproducibility Pipeline
**Run cells sequentially. Pre-computed Drive checkpoints are loaded automatically — completed folds are skipped.**

| Cell | Content |
|---|---|
| 1 | Installation, Drive mount (optional), constants |
| 2 | Data loading (auto-downloaded from GitHub) |
| 3 | Architecture (Dual-Path E-GACT + all baselines) |
| 4 | 5-fold CV — E-GACT + all baselines |
| 5 | Ablation study |
| 6 | Calibration curves |
| 7 | t-SNE latent space visualisation |
| 8 | SHAP feature attribution |
| 9 | Neighbourhood Influence Score (NIS) |
| 10 | Final results table + figure download |


In [ ]:
# ==============================================================================
# CELL 1: INSTALLATION + CONSTANTS
# ==============================================================================
!pip install -q torch_geometric faiss-cpu scikit-learn lightgbm pytorch-tabnet tab-transformer-pytorch shap catboost xgboost

import os, json, glob, urllib.request, zipfile, shutil, time, warnings
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.nn import GCNConv, SAGEConv
import faiss, shap, gc
import matplotlib, matplotlib.pyplot as plt, matplotlib.colors as mcolors
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from scipy import stats
from scipy.stats import pearsonr
from lightgbm import LGBMClassifier
warnings.filterwarnings('ignore')
torch.cuda.empty_cache(); gc.collect()

# ── Checkpoint directory (local session storage) ──────────────────
DRIVE = '/content/EGACT_v4'
os.makedirs(DRIVE, exist_ok=True)
print(f'Checkpoint directory: {DRIVE}')

# ── Local data directory (independent of Drive) ───────────────────
DATA_DIR = '/content/EGACT_data'
os.makedirs(DATA_DIR, exist_ok=True)

# ── GitHub raw base URL ───────────────────────────────────────────
GITHUB_RAW = 'https://github.com/medical-ai-researcher/E-GACT/raw/main'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Hyperparameters and constants
D_MODEL = 32; N_HEADS = 4; DROPOUT = 0.3
K = 10; ALPHA = 0.5; LAMBDA_BCE = 0.6; LAMBDA_SCL = 0.4
TEMP_SCL = 0.07; N_FOLDS = 5; SEED = 42

# Global result stores
E_GACT_Models   = {}
E_GACT_Datasets = {}
CV_RESULTS      = {}
CV_SUMMARY      = {}
SOTA_RESULTS    = {}

NHANES_CODEBOOK = {
    'RIDAGEYR':'Age (Years)', 'RIAGENDR':'Gender', 'RIDRETH1':'Race/Ethnicity',
    'BMXBMI':'BMI', 'BMXWAIST':'Waist Circumference (cm)',
    'BMXHT':'Height (cm)', 'BMXWT':'Weight (kg)',
    'BPXSY1':'Systolic BP 1 (mmHg)', 'BPXDI1':'Diastolic BP 1 (mmHg)',
    'BPQ020':'High Blood Pressure', 'BPQ060':'High Cholesterol',
    'BPQ050A':'Taking BP Medication', 'BPQ090D':'Taking Statin',
    'MCQ160B':'Heart Failure', 'MCQ160C':'Coronary Heart Disease',
    'MCQ160E':'Heart Attack', 'MCQ160F':'Stroke',
    'INDFMPIR':'Poverty Income Ratio', 'INDHHIN2':'Household Income',
}
print('Cell 1 — Installation and constants: OK')


In [ ]:
# ==============================================================================
# CELL 2: DATA LOADING — Auto-download from GitHub
# ==============================================================================
def _download(url, dest, desc='Downloading'):
    if os.path.exists(dest):
        print(f'  Cached: {dest}'); return
    print(f'  {desc}...')
    for attempt in range(5):
        try:
            urllib.request.urlretrieve(url, dest)
            print(f'  OK: {dest}'); return
        except Exception as e:
            print(f'  Attempt {attempt+1}/5 failed: {e}')
            if attempt < 4: time.sleep(10)
    raise RuntimeError(f'Download failed: {url}')

def load_dataset(name, brfss_mode='balanced'):
    print(f'Loading [{name}]...')

    if name == 'NHANES':
        folder = os.path.join(DATA_DIR, 'NHANES')
        if not os.path.exists(folder):
            zp = os.path.join(DATA_DIR, 'NHANES_Dataset_2017.zip')
            _download(
                f'{GITHUB_RAW}/data/NHANES_Dataset_2017.zip',
                zp, 'Downloading NHANES from GitHub'
            )
            with zipfile.ZipFile(zp) as zf:
                zf.extractall(folder)
            print(f'  Extracted to {folder}')
        def rxpt(fn):
            p = os.path.join(folder, fn)
            if not os.path.exists(p): p = p.replace('.xpt','.XPT')
            return pd.read_sas(p) if os.path.exists(p) else None
        dfs = list(map(rxpt,['DEMO_J.xpt','BMX_J.xpt','BPQ_J.xpt',
                              'MCQ_J.xpt','GHB_J.xpt','GLU_J.xpt']))
        df = dfs[0]
        for d in dfs[1:]:
            if d is not None: df = pd.merge(df, d, on='SEQN', how='left')
        df = df.dropna(subset=['LBXGH','LBXGLU'], how='all')
        y  = np.zeros(len(df), dtype=int)
        y[df['LBXGH'] >= 6.5] = 1; y[df['LBXGLU'] >= 126] = 1
        X = df.drop(columns=[c for c in ['SEQN','LBXGH','LBXGLU','LBDGLUSI'] if c in df.columns])
        X = X.loc[:, X.isnull().mean() < 0.4]

    elif name == '130-US':
        cp = os.path.join(DATA_DIR, 'diabetic_data.csv')
        _download(
            f'{GITHUB_RAW}/data/diabetic_data.csv',
            cp, 'Downloading 130-US EHR from GitHub'
        )
        df = pd.read_csv(cp)
        print(f'  N={len(df)}, F={df.shape[1]-1}')
        y = (df['readmitted'] == '<30').astype(int).values
        X = df.drop(columns=[c for c in
            ['encounter_id','patient_nbr','weight','payer_code','readmitted']
            if c in df.columns])

    elif name == 'BRFSS':
        tag  = brfss_mode
        fname = ('diabetes_binary_5050split_health_indicators_BRFSS2015.csv.gz'
                 if tag == 'balanced' else
                 'diabetes_binary_health_indicators_BRFSS2015.csv.gz')
        cp = os.path.join(DATA_DIR, f'BRFSS_{tag}.csv.gz')
        # Try GitHub first; fall back to mirror if unavailable
        try:
            _download(f'{GITHUB_RAW}/data/{fname}', cp, f'Downloading BRFSS {tag} from GitHub')
        except:
            url = ('https://gedeck.github.io/DS-6030/datasets/diabetes/' + fname)
            _download(url, cp, f'Downloading BRFSS {tag} from mirror')
        nmax = 50000 if tag == 'balanced' else 70000
        df = pd.read_csv(cp)
        if len(df) > nmax:
            df = df.sample(nmax, random_state=SEED).reset_index(drop=True)
        y = df['Diabetes_binary'].astype(int).values
        X = df.drop(columns=['Diabetes_binary'])
        print(f'  Source: {tag}, N={len(X)}, F={X.shape[1]}, prevalence={y.mean():.3f}')

    else:
        raise ValueError(f'Unknown dataset: {name}')

    for col in X.select_dtypes(include=['object']).columns:
        try: X[col] = X[col].str.decode('utf-8')
        except: pass

    print(f'  N={len(X)}, F={X.shape[1]}, prevalence={y.mean():.3f}')
    return X.reset_index(drop=True), y

print('Cell 2 — Data loading functions defined. Datasets will be auto-downloaded from GitHub.')


In [ ]:
# ==============================================================================
# CELL 3: ARCHITECTURE — Dual-Path E-GACT + all baseline models
# ==============================================================================

# ── Supervised Contrastive Loss ──────────────────────────────────────────────
class SCLoss(nn.Module):
    def __init__(self, tau=0.07):
        super().__init__(); self.tau = tau
    def forward(self, feat, labels):
        feat = F.normalize(feat, p=2, dim=1)
        sim  = torch.matmul(feat, feat.T) / self.tau
        labels = labels.view(-1, 1)
        mask   = torch.eq(labels, labels.T).float().to(feat.device)
        lmask  = torch.scatter(torch.ones_like(mask), 1,
            torch.arange(feat.shape[0]).view(-1,1).to(feat.device), 0)
        mask = mask * lmask
        mv = torch.max(sim, dim=1, keepdim=True)[0].detach()
        ex = torch.exp(sim - mv) * lmask
        lp = (sim - mv) - torch.log(ex.sum(1, keepdim=True) + 1e-9)
        return -(mask * lp).sum(1) / (mask.sum(1) + 1e-9)

# ── Dual-Path E-GACT Model ───────────────────────────────────────────────────
class NumTok(nn.Module):
    def __init__(self, n, d):
        super().__init__()
        self.w = nn.Parameter(torch.randn(1,n,d)*0.01)
        self.b = nn.Parameter(torch.zeros(1,n,d))
    def forward(self, x): return x.unsqueeze(-1)*self.w+self.b

class DualEGACT(nn.Module):
    def __init__(self, num_cont, cat_dims, num_raw, d=D_MODEL, h=N_HEADS, drop=DROPOUT):
        super().__init__()
        self.ctok  = NumTok(num_cont, d) if num_cont > 0 else None
        self.cemb  = nn.ModuleList([nn.Embedding(c,d) for c in cat_dims]) if cat_dims else None
        self.cls   = nn.Parameter(torch.randn(1,1,d))
        enc        = nn.TransformerEncoderLayer(d,h,d*4,drop,batch_first=True)
        self.tfm   = nn.TransformerEncoder(enc, num_layers=2)
        self.rproj = nn.Sequential(nn.Linear(num_raw,d),nn.LayerNorm(d),nn.ReLU(),nn.Dropout(drop))
        self.gcn1  = GCNConv(d,d); self.gcn2 = GCNConv(d,d)
        self.scl_head = nn.Sequential(nn.Linear(d,d),nn.ReLU(),nn.Linear(d,16))
        self.clf   = nn.Sequential(nn.Linear(d*2,d),nn.LayerNorm(d),nn.ReLU(),nn.Dropout(0.2),nn.Linear(d,1))

    def get_z(self, xc, xk):
        bs = xc.shape[0] if xc is not None else xk.shape[0]
        t  = []
        if self.ctok and xc is not None: t.append(self.ctok(xc))
        if self.cemb and xk is not None:
            t.append(torch.cat([e(xk[:,i]).unsqueeze(1) for i,e in enumerate(self.cemb)],1).mean(1,keepdim=True))
        x = torch.cat(t,1)
        x = torch.cat([self.cls.expand(bs,-1,-1),x],1)
        return self.tfm(x)[:,0,:]

    def get_h(self, xraw, ei):
        h = self.rproj(xraw)
        if ei.shape[1] > 0:
            h = F.gelu(self.gcn1(h,ei)); h = self.gcn2(h,ei)
        return h

    def predict(self, z, h):
        return self.clf(torch.cat([F.normalize(z,2,1),F.normalize(h,2,1)],1))


def build_graph(Xc_src, Xk_src, k=K, alpha=ALPHA, Xc_tgt=None, Xk_tgt=None, inductive=False):
    n = Xc_src.shape[0]
    src = Xc_src.astype('float32').copy()
    nrm = np.linalg.norm(src,axis=1,keepdims=True); nrm[nrm==0]=1; src/=nrm
    idx = faiss.IndexFlatIP(src.shape[1]); idx.add(src)
    qry = src if not inductive else (lambda t: t/np.maximum(np.linalg.norm(t,axis=1,keepdims=True),1e-9))(Xc_tgt.astype('float32').copy())
    n_cand = min(2*k+1, n)
    sims, inds = idx.search(qry, n_cand)
    edges = []
    for i in range(qry.shape[0]):
        scored = []
        for ci, j in enumerate(inds[i]):
            if j<0: continue
            if not inductive and j==i: continue
            cos = float(sims[i,ci])
            jac = 0.0
            if Xk_src is not None and Xk_src.shape[1]>0:
                a = (Xk_tgt[i] if (inductive and Xk_tgt is not None) else Xk_src[i])
                b = Xk_src[j]
                inter=np.sum((a>0)&(b>0)&(a==b)); union=np.sum((a>0)|(b>0))
                jac = float(inter)/float(union) if union>0 else 0.0
            scored.append((alpha*cos+(1-alpha)*jac, j))
        scored.sort(key=lambda x:-x[0])
        src_node = n+i if inductive else i
        for _,j in scored[:k]:
            edges.append([j,src_node]) if inductive else (edges.append([j,i]),edges.append([i,j]))
    if not edges: return torch.zeros((2,0),dtype=torch.long)
    return torch.tensor(list(set(map(tuple,edges))),dtype=torch.long).t().contiguous()


# ── BASELINE MODELLER ─────────────────────────────────────────────────────────
class MLP(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d,256),nn.BatchNorm1d(256),nn.ReLU(),nn.Dropout(0.3),
            nn.Linear(256,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Dropout(0.3),
            nn.Linear(128,1))
    def forward(self,x): return self.net(x)

class SAINTModel(nn.Module):
    def __init__(self, d, hidden=64):
        super().__init__()
        self.proj = nn.Linear(d, hidden)
        enc = nn.TransformerEncoderLayer(hidden,4,256,0.1,batch_first=True)
        self.tfm  = nn.TransformerEncoder(enc, num_layers=3)
        self.cls  = nn.Parameter(torch.randn(1,1,hidden))
        self.head = nn.Sequential(nn.Linear(hidden,32),nn.ReLU(),nn.Linear(32,1))
    def forward(self,x):
        bs = x.shape[0]
        t  = self.proj(x).unsqueeze(1)
        t  = torch.cat([self.cls.expand(bs,-1,-1),t],1)
        return self.head(self.tfm(t)[:,0,:])

class GraphSAGEModel(nn.Module):
    def __init__(self, d, hidden=64):
        super().__init__()
        self.proj  = nn.Linear(d,hidden)
        self.sage1 = SAGEConv(hidden,hidden)
        self.sage2 = SAGEConv(hidden,hidden)
        self.head  = nn.Sequential(nn.Linear(hidden,32),nn.ReLU(),nn.Linear(32,1))
    def forward(self,x,ei):
        h = F.relu(self.proj(x))
        h = F.relu(self.sage1(h,ei))
        h = self.sage2(h,ei)
        return self.head(h)


def _train_simple(model, Xtr, ytt, pw, epochs=100, patience=15):
    opt = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    bce = nn.BCEWithLogitsLoss(pos_weight=pw)
    loader = DataLoader(TensorDataset(Xtr,ytt), batch_size=256, shuffle=True)
    best, no_imp = float('inf'), 0
    for ep in range(epochs):
        model.train()
        for xb,yb in loader:
            opt.zero_grad(); bce(model(xb),yb).backward(); opt.step()
        no_imp += 1
        if no_imp >= patience: break
    return model

def run_mlp(Xc_tr,Xk_tr,y_tr,Xc_va,Xk_va,y_va):
    has_cat = Xk_tr is not None
    Xtr = torch.tensor(np.hstack([Xc_tr,Xk_tr]) if has_cat else Xc_tr,dtype=torch.float32).to(device)
    Xva = torch.tensor(np.hstack([Xc_va,Xk_va]) if has_cat else Xc_va,dtype=torch.float32).to(device)
    ytt = torch.tensor(y_tr,dtype=torch.float32).unsqueeze(1).to(device)
    pw  = torch.tensor([(len(y_tr)-y_tr.sum())/(y_tr.sum()+1e-5)]).to(device)
    model = MLP(Xtr.shape[1]).to(device)
    opt = optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
    bce = nn.BCEWithLogitsLoss(pos_weight=pw)
    loader = DataLoader(TensorDataset(Xtr,ytt),batch_size=256,shuffle=True)
    best_auc,best_state,no_imp = 0.,None,0
    for ep in range(100):
        model.train()
        for xb,yb in loader: opt.zero_grad();bce(model(xb),yb).backward();opt.step()
        model.eval()
        with torch.no_grad(): pr=torch.sigmoid(model(Xva)).cpu().numpy().flatten()
        auc=roc_auc_score(y_va,pr)
        if auc>best_auc: best_auc=auc;best_state={k:v.cpu() for k,v in model.state_dict().items()};no_imp=0
        else: no_imp+=1
        if no_imp>=15: break
    return best_auc

def run_tabnet(Xc_tr,Xk_tr,y_tr,Xc_va,Xk_va,y_va):
    from pytorch_tabnet.tab_model import TabNetClassifier
    has_cat = Xk_tr is not None
    Xtr = np.hstack([Xc_tr,Xk_tr]) if has_cat else Xc_tr
    Xva = np.hstack([Xc_va,Xk_va]) if has_cat else Xc_va
    clf = TabNetClassifier(n_d=16,n_a=16,n_steps=3,verbose=0,
        device_name='cuda' if torch.cuda.is_available() else 'cpu')
    clf.fit(Xtr,y_tr,eval_set=[(Xva,y_va)],eval_metric=['auc'],
            max_epochs=50,patience=10,batch_size=256,virtual_batch_size=128)
    return roc_auc_score(y_va, clf.predict_proba(Xva)[:,1])

def run_fttransformer(Xc_tr,Xk_tr,y_tr,Xc_va,Xk_va,y_va,cat_dims):
    from tab_transformer_pytorch import FTTransformer
    has_cat = Xk_tr is not None and len(cat_dims)>0
    Xct = torch.tensor(Xc_tr,dtype=torch.float32).to(device)
    Xcv = torch.tensor(Xc_va,dtype=torch.float32).to(device)
    ytt = torch.tensor(y_tr,dtype=torch.float32).unsqueeze(1).to(device)
    pw  = torch.tensor([(len(y_tr)-y_tr.sum())/(y_tr.sum()+1e-5)]).to(device)
    Xkt = torch.tensor(Xk_tr,dtype=torch.long).to(device) if has_cat else None
    Xkv = torch.tensor(Xk_va,dtype=torch.long).to(device) if has_cat else None
    model = FTTransformer(categories=tuple(cat_dims) if has_cat else (),
        num_continuous=Xc_tr.shape[1],dim=32,depth=3,heads=4,dim_out=1,
        attn_dropout=0.1,ff_dropout=0.1).to(device)
    opt = optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
    bce = nn.BCEWithLogitsLoss(pos_weight=pw)
    loader = DataLoader(TensorDataset(Xct,ytt),batch_size=256,shuffle=True)
    best_auc,no_imp = 0.,0
    for ep in range(50):
        model.train()
        for bi,(xc,yb) in enumerate(loader):
            opt.zero_grad()
            xk = Xkt[bi*256:(bi+1)*256] if has_cat else torch.zeros((len(xc),0),dtype=torch.long,device=device)
            bce(model(xk,xc),yb).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            xk_va = Xkv if has_cat else torch.zeros((len(Xcv),0),dtype=torch.long,device=device)
            pr = torch.sigmoid(model(xk_va,Xcv)).cpu().numpy().flatten()
        auc = roc_auc_score(y_va,pr)
        if auc>best_auc: best_auc=auc;no_imp=0
        else: no_imp+=1
        if no_imp>=15: break
    return best_auc

def run_saint(Xc_tr,Xk_tr,y_tr,Xc_va,Xk_va,y_va):
    has_cat = Xk_tr is not None
    Xtr = torch.tensor(np.hstack([Xc_tr,Xk_tr]) if has_cat else Xc_tr,dtype=torch.float32).to(device)
    Xva = torch.tensor(np.hstack([Xc_va,Xk_va]) if has_cat else Xc_va,dtype=torch.float32).to(device)
    ytt = torch.tensor(y_tr,dtype=torch.float32).unsqueeze(1).to(device)
    pw  = torch.tensor([(len(y_tr)-y_tr.sum())/(y_tr.sum()+1e-5)]).to(device)
    model = SAINTModel(Xtr.shape[1]).to(device)
    opt = optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
    bce = nn.BCEWithLogitsLoss(pos_weight=pw)
    loader = DataLoader(TensorDataset(Xtr,ytt),batch_size=256,shuffle=True)
    best_auc,best_state,no_imp = 0.,None,0
    for ep in range(100):
        model.train()
        for xb,yb in loader: opt.zero_grad();bce(model(xb),yb).backward();opt.step()
        model.eval()
        with torch.no_grad(): pr=torch.sigmoid(model(Xva)).cpu().numpy().flatten()
        auc=roc_auc_score(y_va,pr)
        if auc>best_auc: best_auc=auc;no_imp=0
        else: no_imp+=1
        if no_imp>=15: break
    return best_auc

def run_graphsage(Xc_tr,Xk_tr,y_tr,Xc_va,Xk_va,y_va):
    has_cat = Xk_tr is not None
    X_raw_tr = np.hstack([Xc_tr,Xk_tr]).astype('float32') if has_cat else Xc_tr.astype('float32')
    X_raw_va = np.hstack([Xc_va,Xk_va]).astype('float32') if has_cat else Xc_va.astype('float32')
    ei_tr = build_graph(Xc_tr,Xk_tr,k=K,alpha=ALPHA).to(device)
    ei_va = build_graph(Xc_tr,Xk_tr,k=K,alpha=ALPHA,Xc_tgt=Xc_va,Xk_tgt=Xk_va,inductive=True).to(device)
    ei_all = torch.cat([ei_tr,ei_va],dim=1)
    X_all  = torch.tensor(np.vstack([X_raw_tr,X_raw_va]),dtype=torch.float32).to(device)
    ytt    = torch.tensor(y_tr,dtype=torch.float32).unsqueeze(1).to(device)
    pw     = torch.tensor([(len(y_tr)-y_tr.sum())/(y_tr.sum()+1e-5)]).to(device)
    model  = GraphSAGEModel(X_raw_tr.shape[1]).to(device)
    opt    = optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
    bce    = nn.BCEWithLogitsLoss(pos_weight=pw)
    n_tr   = len(y_tr)
    best_auc,best_state,no_imp = 0.,None,0
    for ep in range(50):
        model.train(); opt.zero_grad()
        out  = model(X_all,ei_all)
        bce(out[:n_tr],ytt).backward(); opt.step()
        model.eval()
        with torch.no_grad(): pr=torch.sigmoid(out[n_tr:]).cpu().numpy().flatten()
        auc = roc_auc_score(y_va,pr)
        if auc>best_auc: best_auc=auc;no_imp=0
        else: no_imp+=1
        if no_imp>=15: break
    return best_auc

print('Cell 3 — Architecture and all baseline models defined: OK')


In [ ]:
# ==============================================================================
# CELL 4: 5-FOLD STRATIFIED CV — E-GACT + ALL BASELINES
# Skips completed folds if Drive checkpoints exist; resumes from last saved fold.
# ==============================================================================
def run_egact_fold(Xc_tr,Xk_tr,y_tr,Xc_va,Xk_va,y_va,cat_dims,ds,fold,cfg='full'):
    use_gnn = cfg in ('gnn','full')
    use_scl = cfg in ('scl','full')
    has_cat = Xk_tr is not None
    Xct = torch.tensor(Xc_tr,dtype=torch.float32).to(device)
    Xcv = torch.tensor(Xc_va,dtype=torch.float32).to(device)
    ytt = torch.tensor(y_tr,dtype=torch.float32).unsqueeze(1).to(device)
    Xkt = torch.tensor(Xk_tr,dtype=torch.long).to(device) if has_cat else torch.zeros((len(Xc_tr),1),dtype=torch.long,device=device)
    Xkv = torch.tensor(Xk_va,dtype=torch.long).to(device) if has_cat else torch.zeros((len(Xc_va),1),dtype=torch.long,device=device)
    Xr_tr = np.hstack([Xc_tr,Xk_tr]).astype('float32') if has_cat else Xc_tr.astype('float32')
    Xr_va = np.hstack([Xc_va,Xk_va]).astype('float32') if has_cat else Xc_va.astype('float32')
    num_raw = Xr_tr.shape[1]
    if use_gnn:
        print(f'    Building hybrid k-NN graph (k={K})...', end=' ', flush=True)
        ei_tr  = build_graph(Xc_tr,Xk_tr,k=K,alpha=ALPHA).to(device)
        ei_va  = build_graph(Xc_tr,Xk_tr,k=K,alpha=ALPHA,Xc_tgt=Xc_va,Xk_tgt=Xk_va,inductive=True).to(device)
        ei_all = torch.cat([ei_tr,ei_va],dim=1)
        Xr_tr_t = torch.tensor(Xr_tr,dtype=torch.float32).to(device)
        Xr_all  = torch.tensor(np.vstack([Xr_tr,Xr_va]),dtype=torch.float32).to(device)
        print('done.')
    model  = DualEGACT(num_cont=Xc_tr.shape[1],cat_dims=cat_dims,num_raw=num_raw).to(device)
    opt    = optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
    epochs = 100 if ds=='NHANES' else 50
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
    pw     = torch.tensor([(ytt.shape[0]-ytt.sum())/(ytt.sum()+1e-5)]).to(device)
    bce    = nn.BCEWithLogitsLoss(pos_weight=pw)
    scl_fn = SCLoss(TEMP_SCL)
    pat    = 25 if ds=='NHANES' else 15
    best_auc,best_state,best_m,no_imp = 0.,None,{},0
    for ep in range(1,epochs+1):
        model.train()
        for xc,xk,yb in DataLoader(TensorDataset(Xct,Xkt,ytt),batch_size=256,shuffle=True):
            xk2 = xk if has_cat else None
            opt.zero_grad()
            z     = model.get_z(xc,xk2)
            z_scl = model.scl_head(z)
            if use_gnn:
                xr_b  = torch.cat([xc,xk.float()],1) if has_cat else xc
                xr_np = xr_b.cpu().numpy()
                xk_np = xk.cpu().numpy() if has_cat else None
                ei_b  = build_graph(xr_np,xk_np,k=min(5,xr_np.shape[0]-1)).to(device)
                xr_t  = torch.tensor(xr_np,dtype=torch.float32,device=device)
                h     = model.get_h(xr_t,ei_b)
                lo    = model.predict(z,h)
            else:
                lo = model.predict(z,z)
            loss = LAMBDA_BCE*bce(lo,yb)
            if use_scl: loss = loss + LAMBDA_SCL*scl_fn(z_scl,yb).mean()
            loss.backward(); opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            if use_gnn:
                h_all = model.get_h(Xr_all,ei_all)
                h_va  = h_all[len(Xc_tr):]
                z_va  = model.get_z(Xcv,Xkv)
                pr    = torch.sigmoid(model.predict(z_va,h_va)).cpu().numpy().flatten()
            else:
                z_va = model.get_z(Xcv,Xkv)
                pr   = torch.sigmoid(model.predict(z_va,z_va)).cpu().numpy().flatten()
        auc = roc_auc_score(y_va,pr)
        if auc > best_auc:
            best_auc = auc
            best_state = {k:v.cpu() for k,v in model.state_dict().items()}
            no_imp = 0
            best_f1,best_t = 0,0.5
            for t in np.arange(0.2,0.8,0.05):
                f1 = f1_score(y_va,(pr>=t).astype(int),zero_division=0)
                if f1>best_f1: best_f1,best_t=f1,t
            pc = platt(pr,y_va)
            best_m = {'fold':fold,'ep':ep,'cfg':cfg,'auc':float(auc),
                'auprc':float(average_precision_score(y_va,pr)),
                'f1':float(best_f1),'thresh':float(best_t),
                'brier':float(brier_score_loss(y_va,pc)),
                'probs':pr.tolist(),'y_va':y_va.tolist()}
        else: no_imp+=1
        if ep%10==0 or ep==1:
            print(f'    Ep {ep:03d}/{epochs} | AUC:{auc:.4f} | Best:{best_auc:.4f}')
        if no_imp>=pat:
            print(f'    Early stop ep {ep} (best {best_m["ep"]})')
            break
    model.load_state_dict(best_state)
    return best_m, model


DS_LIST = ['NHANES','130-US','BRFSS_balanced','BRFSS_imbalanced']
BASELINE_FNS = {
    'lgbm':         None,  # defined separately below
    'mlp':          run_mlp,
    'tabnet':       run_tabnet,
    'fttransformer':run_fttransformer,
    'saint':        run_saint,
    'graphsage':    run_graphsage,
}

for ds_key in DS_LIST:
    ds_name    = 'BRFSS' if 'BRFSS' in ds_key else ds_key
    brfss_mode = 'imbalanced' if 'imbalanced' in ds_key else 'balanced'
    ckpt_cv    = f'{DRIVE}/{ds_key}_cv.json'
    ckpt_sota  = f'{DRIVE}/{ds_key}_sota.json'

    print(f'\n{"="*60}\n{ds_key}\n{"="*60}')

    # Veri
    X_df, y_all = load_dataset(ds_name, brfss_mode)
    cat_cols  = X_df.select_dtypes(include=['object','category','O']).columns.tolist()
    cont_cols = X_df.select_dtypes(include=['int64','float64']).columns.tolist()
    cat_dims  = []
    if cat_cols:
        ci_f = SimpleImputer(strategy='most_frequent')
        en_f = OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1)
        cat_dims = [int(x)+2 for x in (en_f.fit_transform(ci_f.fit_transform(X_df[cat_cols]))+1).max(0)]
    skf = StratifiedKFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED)

    # ── LightGBM ────────────────────────────────────────────────────────────
    sota = json.load(open(ckpt_sota)) if os.path.exists(ckpt_sota) else {}
    lgbm_aucs,lgbm_briers,lgbm_probs = [],[],[]
    for tr_i,va_i in skf.split(X_df,y_all):
        Xc_tr,Xc_va,Xk_tr,Xk_va = preprocess(X_df.iloc[tr_i],X_df.iloc[va_i],cont_cols,cat_cols)
        Xl = np.hstack([Xc_tr,Xk_tr]) if Xk_tr is not None else Xc_tr
        Xv = np.hstack([Xc_va,Xk_va]) if Xk_va is not None else Xc_va
        lgbm = LGBMClassifier(n_estimators=200,max_depth=7 if ds_name=='130-US' else 5,
                               random_state=42,n_jobs=-1,verbose=-1)
        lgbm.fit(Xl,y_all[tr_i])
        p = lgbm.predict_proba(Xv)[:,1]
        lgbm_aucs.append(roc_auc_score(y_all[va_i],p))
        lgbm_briers.append(brier_score_loss(y_all[va_i],p))
        lgbm_probs.append((p,y_all[va_i]))
    sota['lgbm'] = lgbm_aucs
    print(f'LightGBM: AUC={np.mean(lgbm_aucs):.4f}±{np.std(lgbm_aucs):.4f}')

    # ── Additional baseline models ───────────────────────────────────────────
    for mname, mfn in BASELINE_FNS.items():
        if mname == 'lgbm': continue
        if len(sota.get(mname,[])) >= N_FOLDS:
            print(f'{mname:15s}: checkpoint ✅  AUC={np.mean(sota[mname]):.4f}')
            continue
        aucs = list(sota.get(mname,[]))
        done = len(aucs)
        print(f'{mname:15s}: resuming from fold {done+1}/5...')
        for fi,(tr_i,va_i) in enumerate(skf.split(X_df,y_all),1):
            if fi<=done: continue
            Xc_tr,Xc_va,Xk_tr,Xk_va = preprocess(X_df.iloc[tr_i],X_df.iloc[va_i],cont_cols,cat_cols)
            try:
                if mname=='fttransformer':
                    auc = mfn(Xc_tr,Xk_tr,y_all[tr_i],Xc_va,Xk_va,y_all[va_i],cat_dims)
                else:
                    auc = mfn(Xc_tr,Xk_tr,y_all[tr_i],Xc_va,Xk_va,y_all[va_i])
                aucs.append(float(auc))
                print(f'  fold {fi}: {auc:.4f}')
            except Exception as e:
                aucs.append(0.0); print(f'  fold {fi}: HATA {e}')
            sota[mname] = aucs
            json.dump(sota,open(ckpt_sota,'w'),indent=2)
        print(f'{mname:15s}: AUC={np.mean(aucs):.4f}±{np.std(aucs):.4f}')

    json.dump(sota,open(ckpt_sota,'w'),indent=2)
    SOTA_RESULTS[ds_key] = sota

    # ── E-GACT 5-fold ───────────────────────────────────────────────────────
    fold_results = json.load(open(ckpt_cv)) if os.path.exists(ckpt_cv) else []
    done_folds   = len(fold_results)
    best_model,best_auc_overall = None,0.
    for fold_idx,(tr_i,va_i) in enumerate(skf.split(X_df,y_all),1):
        if fold_idx<=done_folds:
            print(f'  E-GACT fold {fold_idx}/5 — skipped ✅'); continue
        print(f'\n  E-GACT fold {fold_idx}/5 (train={len(tr_i)}, val={len(va_i)})')
        Xc_tr,Xc_va,Xk_tr,Xk_va = preprocess(X_df.iloc[tr_i],X_df.iloc[va_i],cont_cols,cat_cols)
        m,model = run_egact_fold(Xc_tr,Xk_tr,y_all[tr_i],Xc_va,Xk_va,y_all[va_i],
                                 cat_dims,ds_name,fold_idx,cfg='full')
        fold_results.append(m)
        json.dump(fold_results,open(ckpt_cv,'w'),indent=2)
        print(f'  Fold {fold_idx} | AUC={m["auc"]:.4f} | Brier={m["brier"]:.4f} | saved.')
        if m['auc']>best_auc_overall:
            best_auc_overall=m['auc']; best_model=model
            torch.save(model.state_dict(),f'{DRIVE}/{ds_key}_best_model.pt')

    CV_RESULTS[ds_key] = fold_results
    E_GACT_Models[ds_key] = best_model
    E_GACT_Datasets[ds_key] = {'X_df':X_df,'y':y_all,
        'cont_cols':cont_cols,'cat_cols':cat_cols,'cat_dims':cat_dims}

    # ── Summary ──────────────────────────────────────────────────────────────
    aucs = [m['auc'] for m in fold_results]
    CV_SUMMARY[ds_key] = {
        'auc':(np.mean(aucs),np.std(aucs)),
        'auprc':(np.mean([m['auprc'] for m in fold_results]),np.std([m['auprc'] for m in fold_results])),
        'f1':(np.mean([m['f1'] for m in fold_results]),np.std([m['f1'] for m in fold_results])),
        'brier':(np.mean([m['brier'] for m in fold_results]),np.std([m['brier'] for m in fold_results])),
    }
    if ds_key!='BRFSS_balanced':
        eg_p=np.concatenate([np.array(m['probs']) for m in fold_results])
        eg_y=np.concatenate([np.array(m['y_va'])  for m in fold_results])
        lg_p=np.concatenate([p for p,_ in lgbm_probs])
        _,_,z_stat,p_val=delong(eg_y,eg_p,lg_p)
        sig='* (sig.)' if p_val<0.05/3 else '(n.s.)'
    s=CV_SUMMARY[ds_key]
    print(f'\nE-GACT  AUC : {s["auc"][0]:.4f}±{s["auc"][1]:.4f}')
    if ds_key!='BRFSS_balanced':
        print(f'DeLong      : z={z_stat:.3f}, p={p_val:.4f} {sig}')
    print(f'{"="*60}')


In [ ]:
# ==============================================================================
# CELL 5: ABLATION STUDY — fold-level checkpointing
# ==============================================================================
CFG_NAMES = {'base':'Base Transformer','scl':'+SCL (no GNN)',
             'gnn':'+GNN raw (no SCL)','full':'Full E-GACT v4'}
ABLATION = {}

for ds_key in ['NHANES','130-US','BRFSS_imbalanced']:
    ds_name    = 'BRFSS' if 'BRFSS' in ds_key else ds_key
    ckpt_abl   = f'{DRIVE}/{ds_key}_ablation.json'
    done_abl   = json.load(open(ckpt_abl)) if os.path.exists(ckpt_abl) else {}
    d          = E_GACT_Datasets[ds_key]
    X_df,y_all = d['X_df'],d['y']
    cont_cols,cat_cols,cat_dims = d['cont_cols'],d['cat_cols'],d['cat_dims']
    skf = StratifiedKFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED)

    print(f'\n{"="*50}\nABLATION: [{ds_key}]\n{"="*50}')
    cfg_aucs = {c:[] for c in CFG_NAMES}
    for cfg in CFG_NAMES:
        key = f'{ds_key}_{cfg}'
        if key in done_abl and len(done_abl[key])==N_FOLDS:
            cfg_aucs[cfg]=done_abl[key]
            print(f'  {CFG_NAMES[cfg]:28s} checkpoint: {np.mean(cfg_aucs[cfg]):.4f}±{np.std(cfg_aucs[cfg]):.4f}')
            continue
        aucs      = list(done_abl.get(key+'_partial',[]))
        done_folds = len(aucs)
        if done_folds>0: print(f'  {CFG_NAMES[cfg]:28s} resuming from fold {done_folds+1}/5...')
        for fi,(tr_i,va_i) in enumerate(skf.split(X_df,y_all),1):
            if fi<=done_folds: print(f'    fold {fi}: skipped'); continue
            Xc_tr,Xc_va,Xk_tr,Xk_va = preprocess(X_df.iloc[tr_i],X_df.iloc[va_i],cont_cols,cat_cols)
            m,_ = run_egact_fold(Xc_tr,Xk_tr,y_all[tr_i],Xc_va,Xk_va,y_all[va_i],
                                 cat_dims,ds_name,fi,cfg=cfg)
            aucs.append(m['auc'])
            print(f'  {CFG_NAMES[cfg]:28s} fold {fi}: AUC={m["auc"]:.4f}')
            done_abl[key+'_partial']=aucs
            json.dump(done_abl,open(ckpt_abl,'w'),indent=2)
        cfg_aucs[cfg]=aucs
        done_abl[key]=aucs; done_abl.pop(key+'_partial',None)
        json.dump(done_abl,open(ckpt_abl,'w'),indent=2)
    ABLATION[ds_key]=cfg_aucs
    base_m=np.mean(cfg_aucs['base'])
    print(f'\n  Summary [{ds_key}]:')
    for cfg,aucs in cfg_aucs.items():
        print(f'    {CFG_NAMES[cfg]:28s}: {np.mean(aucs):.4f}±{np.std(aucs):.4f} (Δ={np.mean(aucs)-base_m:+.4f})')


In [ ]:
# CELL 6: CALIBRATION CURVES (Platt-scaled reliability diagrams)
fig,axes=plt.subplots(1,3,figsize=(15,5),dpi=150)
for ax,dk,lb in zip(axes,['NHANES','130-US','BRFSS_imbalanced'],
    ['NHANES (Lab T2DM)','130-US Hospitals','BRFSS (Imbalanced 13.9%)']):
    all_p,all_y=[],[]
    for m in CV_RESULTS[dk]:
        pc=platt(np.array(m['probs']),np.array(m['y_va']))
        all_p.append(pc);all_y.append(np.array(m['y_va']))
    pr=np.concatenate(all_p);yc=np.concatenate(all_y)
    bs=brier_score_loss(yc,pr); fp,mp=calibration_curve(yc,pr,n_bins=10)
    ax.plot(mp,fp,'s-',color='#1f77b4',label=f'E-GACT (Brier={bs:.3f})')
    ax.plot([0,1],[0,1],'k--',alpha=0.5,label='Perfect')
    ax.set_xlabel('Mean predicted probability',fontsize=11)
    ax.set_ylabel('Fraction of positives',fontsize=11)
    ax.set_title(lb,fontsize=11,fontweight='bold')
    ax.legend(fontsize=9);ax.set_xlim([0,1]);ax.set_ylim([0,1])
    sns.despine(ax=ax)
plt.suptitle('E-GACT v4 Calibration',fontsize=13,fontweight='bold',y=1.02)
plt.tight_layout()
plt.savefig('fig_calibration.pdf',format='pdf',bbox_inches='tight')
print('Calibration figure saved: fig_calibration.pdf')
plt.show()


In [ ]:
# CELL 7: t-SNE LATENT SPACE VISUALISATION — loads model from Drive if available
def ensure_model(ds_key):
    if E_GACT_Models.get(ds_key) is not None: return
    pt = f'{DRIVE}/{ds_key}_best_model.pt'
    d  = E_GACT_Datasets[ds_key]
    cont_cols,cat_cols,cat_dims = d['cont_cols'],d['cat_cols'],d['cat_dims']
    X_df,y_all = d['X_df'],d['y']
    skf = StratifiedKFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED)
    tr_i,_ = next(skf.split(X_df,y_all))
    Xc_tr,_,Xk_tr,_ = preprocess(X_df.iloc[tr_i],X_df.iloc[tr_i],cont_cols,cat_cols)
    has_cat = Xk_tr is not None
    X_raw = np.hstack([Xc_tr,Xk_tr]) if has_cat else Xc_tr
    model = DualEGACT(num_cont=Xc_tr.shape[1],cat_dims=cat_dims,
                      num_raw=X_raw.shape[1]).to(device)
    if os.path.exists(pt):
        model.load_state_dict(torch.load(pt,map_location=device))
        print(f'{ds_key}: model weights loaded from Drive checkpoint.')
    else:
        print(f'{ds_key}: no .pt checkpoint found — training fold-1 for visualisation...')
        m,model = run_egact_fold(Xc_tr,Xk_tr,y_all[tr_i],Xc_tr,Xk_tr,y_all[tr_i],
                                 cat_dims,ds_key if 'BRFSS' not in ds_key else 'BRFSS',1,'full')
    model.eval(); E_GACT_Models[ds_key] = model

def make_tsne(dk, max_n, n_iter):
    ensure_model(dk)
    model=E_GACT_Models[dk]; model.eval()
    d=E_GACT_Datasets[dk]; X_df,y_all=d['X_df'],d['y']
    cont_cols,cat_cols=d['cont_cols'],d['cat_cols']
    skf=StratifiedKFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED)
    tr_i,va_i=next(skf.split(X_df,y_all))
    Xc_tr,Xc_va,Xk_tr,Xk_va=preprocess(X_df.iloc[tr_i],X_df.iloc[va_i],cont_cols,cat_cols)
    has_cat=Xk_tr is not None
    Xc_all=np.vstack([Xc_tr,Xc_va])
    Xr_all=np.hstack([Xc_all,np.vstack([Xk_tr,Xk_va])]) if has_cat else Xc_all
    Xct=torch.tensor(Xc_all,dtype=torch.float32).to(device)
    Xkt=(torch.tensor(np.vstack([Xk_tr,Xk_va]),dtype=torch.long).to(device)
         if has_cat else torch.zeros((len(Xc_all),1),dtype=torch.long,device=device))
    ei_tr=build_graph(Xc_tr,Xk_tr,k=K).to(device)
    ei_va=build_graph(Xc_tr,Xk_tr,k=K,Xc_tgt=Xc_va,Xk_tgt=Xk_va,inductive=True).to(device)
    Xr_t=torch.tensor(Xr_all,dtype=torch.float32).to(device)
    with torch.no_grad():
        Z=model.get_z(Xct,Xkt)
        H=model.get_h(Xr_t,torch.cat([ei_tr,ei_va],1))
        emb=torch.cat([F.normalize(Z,2,1),F.normalize(H,2,1)],1).cpu().numpy()
    y_e=y_all
    if len(emb)>max_n:
        idx=np.random.RandomState(42).choice(len(emb),max_n,replace=False)
        emb,y_e=emb[idx],y_e[idx]
    tsne=TSNE(n_components=2,perplexity=30,learning_rate='auto',
              max_iter=n_iter,init='pca',random_state=42)
    emb2d=tsne.fit_transform(emb)
    df_t=pd.DataFrame({'Dim-1':emb2d[:,0],'Dim-2':emb2d[:,1]})
    df_t['Clinical Profile']=pd.Series(y_e).map(
        {0:'Healthy (Target=0)',1:'Diabetic/High Risk (Target=1)'}).values
    pal={'Healthy (Target=0)':'#2ca02c','Diabetic/High Risk (Target=1)':'#d62728'}
    sns.set_style('whitegrid')
    plt.figure(figsize=(8,6),dpi=300)
    sns.scatterplot(data=df_t,x='Dim-1',y='Dim-2',hue='Clinical Profile',
                    palette=pal,alpha=0.65,s=28,edgecolor='white',linewidth=0.2)
    plt.xlabel('Dim-1',fontsize=14,fontweight='bold')
    plt.ylabel('Dim-2',fontsize=14,fontweight='bold')
    plt.legend(title='Clinical Profile',fontsize=11,title_fontsize=12,loc='lower right',frameon=True)
    sns.despine(); plt.tight_layout()
    fn=f'fig_tsne_{dk}.pdf'
    plt.savefig(fn,format='pdf',bbox_inches='tight')
    print(f't-SNE saved: {fn}'); plt.close()

make_tsne('NHANES',3000,1000)
make_tsne('130-US',5000,400)
make_tsne('BRFSS_balanced',3000,400)


In [ ]:
# CELL 8: SHAP
def feat_names(dk, cont_cols):
    if 'NHANES' in dk: return [NHANES_CODEBOOK.get(c,c) for c in cont_cols]
    return list(cont_cols)

def run_shap(dk, n_bg=100, n_pts=50, nsamp=100):
    ensure_model(dk)
    print(f'SHAP: {dk}')
    model=E_GACT_Models[dk]; model.eval()
    d=E_GACT_Datasets[dk]; X_df,y_all=d['X_df'],d['y']
    cont_cols,cat_cols=d['cont_cols'],d['cat_cols']
    skf=StratifiedKFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED)
    tr_i,va_i=next(skf.split(X_df,y_all))
    Xc_tr,Xc_va,Xk_tr,Xk_va=preprocess(X_df.iloc[tr_i],X_df.iloc[va_i],cont_cols,cat_cols)
    has_cat=Xk_tr is not None
    Xct=torch.tensor(Xc_tr,dtype=torch.float32).to(device)
    Xcv=torch.tensor(Xc_va,dtype=torch.float32).to(device)
    Xkt=torch.tensor(Xk_tr,dtype=torch.long).to(device) if has_cat else torch.zeros((len(Xc_tr),1),dtype=torch.long,device=device)
    Xkv=torch.tensor(Xk_va,dtype=torch.long).to(device) if has_cat else torch.zeros((len(Xc_va),1),dtype=torch.long,device=device)
    Xr_all=np.vstack([np.hstack([Xc_tr,Xk_tr]) if has_cat else Xc_tr,
                       np.hstack([Xc_va,Xk_va]) if has_cat else Xc_va])
    ei_tr=build_graph(Xc_tr,Xk_tr,k=K).to(device)
    ei_va=build_graph(Xc_tr,Xk_tr,k=K,Xc_tgt=Xc_va,Xk_tgt=Xk_va,inductive=True).to(device)
    Xr_t=torch.tensor(Xr_all,dtype=torch.float32).to(device)
    with torch.no_grad():
        H_all=model.get_h(Xr_t,torch.cat([ei_tr,ei_va],1))
        H_va_f=H_all[len(Xc_tr):]
    rng=np.random.RandomState(42)
    bg=Xct[rng.choice(len(Xct),n_bg,replace=False)].cpu().numpy()
    all_sv=[]
    for ti in range(n_pts):
        def predict(xb):
            xt=torch.tensor(xb,dtype=torch.float32,device=device)
            out=np.zeros(len(xb))
            with torch.no_grad():
                for i in range(len(xb)):
                    xk_s=Xkv[ti:ti+1] if has_cat else None
                    z_p=model.get_z(xt[i:i+1],xk_s)
                    out[i]=model.predict(z_p,H_va_f[ti:ti+1]).item()
            return out
        ex=shap.KernelExplainer(predict,bg)
        sv=ex.shap_values(Xcv[ti:ti+1].cpu().numpy(),nsamples=nsamp,silent=True)
        if isinstance(sv,list): sv=sv[0]
        all_sv.append(sv[0])
        if (ti+1)%10==0: print(f'  {ti+1}/{n_pts}')
    shap_m=np.array(all_sv); test_m=Xcv[:n_pts].cpu().numpy()
    fn_lst=feat_names(dk,cont_cols)
    cmap_c=mcolors.LinearSegmentedColormap.from_list('GR',['#2ca02c','#d62728'])
    fig,ax=plt.subplots(figsize=(9,7),dpi=300)
    shap.summary_plot(shap_m,test_m,feature_names=fn_lst,max_display=10,show=False,cmap=cmap_c)
    ax=plt.gca()
    for cl in ax.collections:
        if isinstance(cl,matplotlib.collections.PathCollection): cl.set_cmap(cmap_c)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.title(f'SHAP — {dk}',fontsize=12,fontweight='bold',pad=10)
    plt.xlabel('SHAP Impact',fontsize=13,fontweight='bold')
    plt.subplots_adjust(left=0.40,right=0.85,top=0.88,bottom=0.12)
    fn=f'fig_shap_{dk}.pdf'; plt.savefig(fn,format='pdf')
    print(f'  Saved: {fn}'); plt.close()

for ds in ['NHANES','130-US','BRFSS_balanced']:
    run_shap(ds)
print('All SHAP done.')


In [ ]:
# CELL 9: NIS
nis_res={}
for dk in ['NHANES','130-US','BRFSS_balanced']:
    ensure_model(dk)
    model=E_GACT_Models[dk]; model.eval()
    d=E_GACT_Datasets[dk]; X_df,y_all=d['X_df'],d['y']
    cont_cols,cat_cols=d['cont_cols'],d['cat_cols']
    skf=StratifiedKFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED)
    tr_i,va_i=next(skf.split(X_df,y_all))
    Xc_tr,Xc_va,Xk_tr,Xk_va=preprocess(X_df.iloc[tr_i],X_df.iloc[va_i],cont_cols,cat_cols)
    has_cat=Xk_tr is not None
    Xct=torch.tensor(Xc_tr,dtype=torch.float32).to(device)
    Xcv=torch.tensor(Xc_va,dtype=torch.float32).to(device)
    Xkt=torch.tensor(Xk_tr,dtype=torch.long).to(device) if has_cat else torch.zeros((len(Xc_tr),1),dtype=torch.long,device=device)
    Xkv=torch.tensor(Xk_va,dtype=torch.long).to(device) if has_cat else torch.zeros((len(Xc_va),1),dtype=torch.long,device=device)
    Xr_all=np.vstack([np.hstack([Xc_tr,Xk_tr]) if has_cat else Xc_tr,
                       np.hstack([Xc_va,Xk_va]) if has_cat else Xc_va])
    ei_tr=build_graph(Xc_tr,Xk_tr,k=K).to(device)
    ei_va=build_graph(Xc_tr,Xk_tr,k=K,Xc_tgt=Xc_va,Xk_tgt=Xk_va,inductive=True).to(device)
    Xr_t=torch.tensor(Xr_all,dtype=torch.float32).to(device)
    y_va=y_all[va_i]
    with torch.no_grad():
        Z_tr=model.get_z(Xct,Xkt); Z_va=model.get_z(Xcv,Xkv)
        H_all=model.get_h(Xr_t,torch.cat([ei_tr,ei_va],1))
        H_tr=H_all[:len(tr_i)]; H_va=H_all[len(tr_i):]
        p_tr=torch.sigmoid(model.predict(Z_tr,H_tr)).cpu().numpy().flatten()
        p_va=torch.sigmoid(model.predict(Z_va,H_va)).cpu().numpy().flatten()
    xtr=Xc_tr.astype('float32'); xva=Xc_va.astype('float32')
    for arr in [xtr,xva]:
        n=np.linalg.norm(arr,axis=1,keepdims=True); n[n==0]=1; arr/=n
    fi=faiss.IndexFlatIP(xtr.shape[1]); fi.add(xtr)
    _,nbr=fi.search(xva,K)
    nis=np.array([p_va[i]-p_tr[nbr[i]].mean() for i in range(len(p_va))])
    err=np.abs(p_va-y_va)
    r,p_r=pearsonr(np.abs(nis),err)
    hi=err[np.abs(nis)>0.05]; lo_e=err[np.abs(nis)<=0.05]
    _,p_mw=mannwhitneyu(hi,lo_e,alternative='less') if len(hi)>0 and len(lo_e)>0 else (0,1)
    nis_res[dk]={'mean_abs':float(np.abs(nis).mean()),'std':float(np.abs(nis).std()),
                 'high_pct':float((np.abs(nis)>0.05).mean()*100),
                 'r':float(r),'p_r':float(p_r),'mw_p':float(p_mw)}
    print(f'{dk}: |NIS|={nis_res[dk]["mean_abs"]:.3f}, High={nis_res[dk]["high_pct"]:.1f}%, r={r:.3f} (p={p_r:.4f})')


In [ ]:
# CELL 10: FINAL RESULTS TABLE + FIGURE DOWNLOAD
print('\n'+'='*70)
print('FINAL RESULTS — E-GACT v4 Dual-Path Architecture')
print('='*70)
MODEL_NAMES = {'lgbm':'LightGBM','mlp':'MLP','tabnet':'TabNet',
               'fttransformer':'FT-Transformer','saint':'SAINT','graphsage':'GraphSAGE'}
for dk in CV_SUMMARY:
    s=CV_SUMMARY[dk]; sota=SOTA_RESULTS.get(dk,{})
    print(f'\n{dk}')
    print(f'  E-GACT    : AUC={s["auc"][0]:.4f}±{s["auc"][1]:.4f} | Brier={s["brier"][0]:.4f}')
    for m,nm in MODEL_NAMES.items():
        if m in sota and sota[m]:
            v=sota[m]
            print(f'  {nm:15s}: AUC={np.mean(v):.4f}±{np.std(v):.4f}')

import glob
figs=sorted(glob.glob('fig_*.pdf'))
print(f'\nFigures: {len(figs)}')
for f in figs: print(f'  {f}')
try:
    from google.colab import files
    for f in figs: files.download(f)
except: print('Auto-download not available outside Colab — figures saved to working directory.')
